In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# -------------------------------------------------------------
# 1. Descarga y Carga del Dataset desde UCI
# -------------------------------------------------------------
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/auto-mpg/auto-mpg.data"

# Nombres oficiales de las columnas según la documentación de UCI
column_names = [
    "mpg",
    "cylinders",
    "displacement",
    "horsepower",
    "weight",
    "acceleration",
    "model_year",
    "origin",
    "car_name"
]

# El archivo usa espacios en blanco como delimitador (delim_whitespace=True)
df = pd.read_csv(
    url,
    names=column_names,
    sep=r"\s+",
    na_values="?"  # Los valores faltantes están codificados con '?'
)

# -------------------------------------------------------------
# 2. Limpieza de Datos y Preparación de Variables
# -------------------------------------------------------------
# Eliminar registros con valores nulos (originalmente 6 filas en 'horsepower')
df = df.dropna().reset_index(drop=True)
df["horsepower"] = df["horsepower"].astype(float)

# Convertir el consumo continuo en un problema de clasificación binaria:
# 1 = Alta eficiencia (mpg > mediana), 0 = Baja eficiencia (mpg <= mediana)
mediana_mpg = df["mpg"].median()
df["high_efficiency"] = (df["mpg"] > mediana_mpg).astype(int)

# Seleccionar variables explicativas numéricas (excluyendo el nombre del auto y target)
features = [
    "cylinders",
    "displacement",
    "horsepower",
    "weight",
    "acceleration",
    "model_year",
    "origin"
]

X = df[features]
y = df["high_efficiency"]

# -------------------------------------------------------------
# 3. Partición de Datos (Train / Test) y Escalado
# -------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# La regresión logística requiere escalado debido a la regularización (L2 por defecto)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# -------------------------------------------------------------
# 4. Entrenamiento del Modelo de Regresión Logística
# -------------------------------------------------------------
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)

# -------------------------------------------------------------
# 5. Evaluación del Modelo
# -------------------------------------------------------------
y_pred = model.predict(X_test_scaled)
y_proba = model.predict_proba(X_test_scaled)[:, 1]

print("=== Reporte de Clasificación ===")
print(classification_report(y_test, y_pred, target_names=["Baja Eficiencia (0)", "Alta Eficiencia (1)"]))

print("ROC-AUC Score:", round(roc_auc_score(y_test, y_proba), 4))

print("\n=== Matriz de Confusión ===")
matriz = confusion_matrix(y_test, y_pred)
df_cm = pd.DataFrame(
    matriz, 
    index=["Real: Baja", "Real: Alta"], 
    columns=["Predicho: Baja", "Predicho: Alta"]
)
print(df_cm)

# -------------------------------------------------------------
# 6. Coeficientes del Modelo (Importancia e Interpretación)
# -------------------------------------------------------------
coef_df = pd.DataFrame({
    "Característica": features,
    "Coeficiente": clf.coef_[0]
}).sort_values(by="Coeficiente", ascending=False)

print("\n=== Coeficientes de Regresión Logística ===")
print(coef_df.to_string(index=False))

=== Reporte de Clasificación ===
                     precision    recall  f1-score   support

Baja Eficiencia (0)       0.98      0.92      0.95        49
Alta Eficiencia (1)       0.92      0.98      0.95        49

           accuracy                           0.95        98
          macro avg       0.95      0.95      0.95        98
       weighted avg       0.95      0.95      0.95        98

ROC-AUC Score: 0.9954

=== Matriz de Confusión ===
            Predicho: Baja  Predicho: Alta
Real: Baja              45               4
Real: Alta               1              48

=== Coeficientes de Regresión Logística ===
Característica  Coeficiente
    model_year     1.073735
        origin     0.178109
  acceleration    -0.044857
     cylinders    -0.120470
  displacement    -0.648000
    horsepower    -1.410441
        weight    -2.019542
